In [1]:
#pip install openpyxl

In [1]:
import urllib.request
import cv2
import numpy as np
from keras.models import load_model
from datetime import datetime
from openpyxl import Workbook, load_workbook
import os

# -----------------------------
# Load Face Detector & Model
# -----------------------------
classifier = cv2.CascadeClassifier(r"C:\Users\Sonali\Desktop\project1\project_face_detect\project_face_detect\haarcascade_frontalface_default.xml")
model = load_model(r"C:\Users\Sonali\Desktop\project1\project_face_detect\project_face_detect\final_model.h5")

URL = "http://10.72.47.231:8080/shot.jpg"

labels = ['khetriya', 'sonali', 'trupti']

# -----------------------------
# Excel File Setup
# -----------------------------

excel_file = "attendance.xlsx"

if not os.path.exists(excel_file):
    wb = Workbook()
    ws = wb.active
    ws.title = "Attendance"
    ws.append(["Name", "Entry Time", "Exit Time", "Total Stay"])
    wb.save(excel_file)

# -----------------------------
# Attendance Dictionary
# -----------------------------
attendance = {}

# Number of frames after which a person is considered exited
EXIT_THRESHOLD = 30

# -----------------------------
# Functions
# -----------------------------
def get_pred_label(pred):
    return labels[pred]


def preprocess(img):
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, (100, 100))
    img = cv2.equalizeHist(img)
    img = img.reshape(1, 100, 100, 1)
    img = img / 255.0
    return img


def save_to_excel(name, entry, exit_time, duration):
    wb = load_workbook(excel_file)
    ws = wb.active
    ws.append([name, entry, exit_time, duration])
    wb.save(excel_file)


# -----------------------------
# Main Loop
# -----------------------------
while True:

    img_url = urllib.request.urlopen(URL)
    image = np.array(bytearray(img_url.read()), np.uint8)
    frame = cv2.imdecode(image, -1)

    detected_people = []

    faces = classifier.detectMultiScale(frame, 1.5, 5)

    for (x, y, w, h) in faces:

        face = frame[y:y+h, x:x+w]

        prediction = np.argmax(model.predict(preprocess(face), verbose=0))
        person = get_pred_label(prediction)

        detected_people.append(person)

        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(frame, person,
                    (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 255, 0),
                    2)

        # Entry
        if person not in attendance:
            attendance[person] = {
                "entry": datetime.now(),
                "last_seen": datetime.now(),
                "inside": True
            }
            print(f"{person} Entered")

        else:
            attendance[person]["last_seen"] = datetime.now()

    # Check Exit
    current_time = datetime.now()

    for person in list(attendance.keys()):

        if attendance[person]["inside"]:

            seconds_missing = (
                current_time - attendance[person]["last_seen"]
            ).total_seconds()

            if seconds_missing > EXIT_THRESHOLD:

                attendance[person]["inside"] = False

                entry = attendance[person]["entry"]
                exit_time = attendance[person]["last_seen"]

                duration = exit_time - entry

                save_to_excel(
                    person,
                    entry.strftime("%Y-%m-%d %H:%M:%S"),
                    exit_time.strftime("%Y-%m-%d %H:%M:%S"),
                    str(duration)
                )

                print(f"{person} Exited")

                del attendance[person]

    cv2.imshow("Attendance System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

trupti Entered


error: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:973: error: (-215:Assertion failed) size.width>0 && size.height>0 in function 'cv::imshow'
